# Week 7 Lab：CTMC、τ-leaping 與 remasking

## 學習目標
- 把 absorbing mask schedule 寫成連續時間 reveal rate。
- 觀察 τ-leaping 步長對 masked fraction 的離散誤差。
- 用 parity oracle 分辨一次 factorized reveal 與 iterative remasking。

> **誠實註記**：token probabilities 由小型 parity 狀態空間精確列舉；`noisy_probs` 是明確標示的人工 surrogate，不是模型 checkpoint。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 707
rng = np.random.default_rng(SEED)
L, MASK = 8, -1
states = ((np.arange(2 ** L)[:, None] >> np.arange(L)) & 1).astype(int)
states = states[states.sum(axis=1) % 2 == 0]

def exact_probs(xt):
    xt = np.asarray(xt)
    observed = xt != MASK
    candidates = states[np.all((states == xt) | (~observed), axis=1)]
    p1 = candidates.mean(axis=0)
    return p1

def noisy_probs(xt, error=.12):
    # An explicit imperfect surrogate: shrink the exact oracle toward 1/2.
    return (1 - error) * exact_probs(xt) + error * .5

example = np.array([1, 0, 1, 1, MASK, MASK, MASK, MASK])
print('exact p(token=1):', exact_probs(example))
print('surrogate p(token=1):', noisy_probs(example))

In [ ]:
def tau_leap_mask_fraction(n_steps, n_tokens=30000):
    # Reverse time goes from t=1 (all masked) toward t=0.02.
    ts = np.linspace(1.0, .02, n_steps + 1)
    masked = np.ones(n_tokens, dtype=bool)
    fractions = [1.0]
    for t_hi, t_lo in zip(ts[:-1], ts[1:]):
        tau = t_hi - t_lo
        p_reveal = 1 - np.exp(-tau / t_hi)  # freeze rate u_t=1/t at step start
        masked &= rng.random(n_tokens) >= p_reveal
        fractions.append(masked.mean())
    return ts, np.array(fractions)

fig, ax = plt.subplots(figsize=(6, 4))
fine_t = np.linspace(1, .02, 200)
ax.plot(fine_t, fine_t, 'k--', label='exact masked fraction = t')
for n in [4, 12, 48]:
    t, frac = tau_leap_mask_fraction(n)
    ax.plot(t, frac, 'o-', ms=3, label=f'{n} τ-leaps')
ax.set(xlabel='reverse-time state t', ylabel='masked fraction', title='τ-leaping freezes the CTMC rate inside each step')
ax.invert_xaxis()
ax.legend()
plt.show()

In [ ]:
def one_shot(error=.12):
    xt = np.full(L, MASK)
    p = noisy_probs(xt, error)
    return (rng.random(L) < p).astype(int)

def remask_sample(rounds=5, error=.12):
    xt = np.full(L, MASK)
    for r in range(rounds):
        hidden = np.flatnonzero(xt == MASK)
        p = noisy_probs(xt, error)
        proposal = xt.copy()
        proposal[hidden] = (rng.random(len(hidden)) < p[hidden]).astype(int)
        confidence = np.maximum(p, 1 - p)
        keep_total = int(np.ceil((r + 1) / rounds * L))
        # Keep observed tokens plus the most confident new proposals; re-mask the rest.
        observed = np.flatnonzero(xt != MASK)
        slots = max(0, keep_total - len(observed))
        order = hidden[np.argsort(-confidence[hidden])]
        xt[order[:slots]] = proposal[order[:slots]]
    # Any remaining token is sampled from the latest conditional.
    hidden = np.flatnonzero(xt == MASK)
    if len(hidden):
        p = noisy_probs(xt, error)
        xt[hidden] = (rng.random(len(hidden)) < p[hidden]).astype(int)
    return xt

def valid(x):
    return np.sum(x) % 2 == 0

rounds = np.array([1, 2, 3, 5, 8])
rates = []
for r in rounds:
    samples = np.array([one_shot() if r == 1 else remask_sample(r) for _ in range(1500)])
    rates.append(np.mean([valid(x) for x in samples]))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rounds, rates, 'o-')
ax.axhline(1, color='k', ls='--', alpha=.4)
ax.set(xlabel='prediction / remasking rounds', ylabel='valid parity rate', ylim=(0, 1.05),
       title='Remasking lets late conditionals repair global structure')
plt.show()

In [ ]:
errors = [0.0, .08, .2, .35]
fig, ax = plt.subplots(figsize=(6, 4))
for error in errors:
    curve = []
    for r in rounds:
        batch = [one_shot(error) if r == 1 else remask_sample(r, error) for _ in range(600)]
        curve.append(np.mean([valid(x) for x in batch]))
    ax.plot(rounds, curve, 'o-', label=f'surrogate error={error}')
ax.set(xlabel='rounds', ylabel='valid parity rate', ylim=(0, 1.05), title='Remasking cannot erase arbitrary model error')
ax.legend(fontsize=8)
plt.show()

## 讀者練習 / TODO
目前 remasking 依 confidence 排序。改成每輪隨機保留同樣數量的 token，比較兩者在 `error=.2` 時的合法率；再說明 parity toy 為什麼會讓 confidence 在早期幾乎全部相同。